In [1]:
# Imports declaration
import os

import numpy as np
import pandas as pd

import joblib

import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split

In [2]:
# Load csvs from datasets/ and combines them into 1 dataframe df
def load_data():
    df_list = []
    
    for file in os.listdir("datasets"):
        if file.endswith(".csv"):
            path = os.path.join("datasets", file)
            df = pd.read_csv(path)
            df.columns = df.columns.str.strip()
            df.dropna(how='all', inplace=True)
            df['Label'] = df['Label'].astype(str).str.strip()
            df_list.append(df)
    
    df = pd.concat(df_list, ignore_index=True)
    return df

df = load_data()

In [ ]:
# Delete unused features
non_feature_cols = ['Flow ID', 'Source IP', 'Destination IP', 'Timestamp', 'Label']
drop_cols = [col for col in non_feature_cols if col in df.columns]
feature_df = df.drop(columns=drop_cols, errors='ignore')

feature_df = feature_df.apply(pd.to_numeric, errors='coerce')
feature_df.replace([np.inf, -np.inf], np.nan, inplace=True)
feature_df.dropna(inplace=True)

# Align label DataFrame with cleaned rows
df = df.loc[feature_df.index]

# Normalize features
scaler = MinMaxScaler()
feature_df_scaled = pd.DataFrame(scaler.fit_transform(feature_df), columns=feature_df.columns)

# Add label column back
feature_df_scaled['Label'] = df['Label'].values

output_path = os.path.join("working", "processed_data.csv")
feature_df_scaled.to_csv(output_path, index=False)
print(f"✅ Processed data saved to: {output_path}")

joblib.dump(scaler, "working/min_max_scaler.pkl")
print(f"✅ MinMaxScaler saved to: working/min_max_scaler.pkl")

In [ ]:
# Autoencoder class declaration
class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super(Autoencoder, self).__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.LeakyReLU(0.1),
            nn.Linear(128, 64),
            nn.LeakyReLU(0.1),
            nn.Linear(64, 32)
        )
        self.decoder = nn.Sequential(
            nn.Linear(32, 64),
            nn.LeakyReLU(0.1),
            nn.Linear(64, 128),
            nn.LeakyReLU(0.1),
            nn.Linear(128, input_dim)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

In [ ]:
# Create training and testing datasets
# feature_df_scaled = pd.read_csv('working/processed_data.csv') # use the one from above
benign_df = feature_df_scaled[feature_df_scaled['Label'] == 'BENIGN'].drop(columns=['Label'])
anomalous_df = feature_df_scaled[feature_df_scaled['Label'] != 'BENIGN']

X_benign_np = benign_df.values.astype('float32')
X_anomalies_np = anomalous_df.drop(columns=['Label']).values.astype('float32')

X_train_benign_subset, X_test_benign_subset = train_test_split(X_benign_np, test_size=0.2, random_state=42)

scaler = joblib.load("working/min_max_scaler.pkl") # saved from parser
X_train_benign_scaled = scaler.transform(X_train_benign_subset)
X_test_benign_scaled = scaler.transform(X_test_benign_subset)
X_test_anomalies_scaled = scaler.transform(X_anomalies_np)

In [ ]:
n_components = 40 # <--- EXPERIMENT WITH THIS VALUE (e.g., 20, 30, 50, 60)
pca = PCA(n_components=n_components, random_state=42)
pca.fit(X_train_benign_scaled)
print(f"\n✅ PCA fitted on {X_train_benign_scaled.shape[1]} features.")
print(f"   Explained variance ratio (first {n_components} components): {pca.explained_variance_ratio_.sum():.4f}")

In [ ]:
X_train_benign_pca = pca.transform(X_train_benign_scaled)
X_test_benign_pca = pca.transform(X_test_benign_scaled)
X_test_anomalies_pca = pca.transform(X_test_anomalies_scaled)
joblib.dump(pca, "working/pca_model.pkl")
print("✅ PCA model saved to: working/pca_model.pkl")

In [ ]:
X_test_eval_np = np.concatenate((X_test_benign_pca, X_test_anomalies_pca), axis=0)

y_test_benign_labels = np.zeros(len(X_test_benign_pca))
y_test_anomaly_labels = np.ones(len(X_test_anomalies_pca))
y_test_eval = np.concatenate((y_test_benign_labels, y_test_anomaly_labels), axis=0)

# Convert to PyTorch tensors (ensure float32)
X_train_benign_torch = torch.tensor(X_train_benign_pca, dtype=torch.float32)
X_test_eval_torch = torch.tensor(X_test_eval_np, dtype=torch.float32)
y_test_eval_torch = torch.tensor(y_test_eval, dtype=torch.long)

In [ ]:
# Update input_dim for Autoencoder to be the number of PCA components
input_dim = X_train_benign_torch.shape[1] # This will now be n_components

model = Autoencoder(input_dim) # Autoencoder now expects n_components as input
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
train_dataset = TensorDataset(X_train_benign_torch)
batch_size = 128 # You can adjust this
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

EPOCHS = 40 # Start with 40, potentially increase after
print(f"\nStarting training with {EPOCHS} epochs, batch_size={batch_size}...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch_idx, (data,) in enumerate(train_loader):
        outputs = model(data)
        loss = criterion(outputs, data)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}/{EPOCHS}, Avg Loss: {avg_loss:.4f}")

In [7]:
# for batch training
train_dataset = TensorDataset(X_train_benign_torch)
batch_size = 128  # experiment with 64, 256, 512
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

input_dim = X_train_benign_torch.shape[1]

model = Autoencoder(input_dim)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 40
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch_idx, (data,) in enumerate(train_loader):  # (data,) because TensorDataset returns a tuple
        outputs = model(data)
        loss = criterion(outputs, data)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}/{EPOCHS}, Loss: {avg_loss:.4f}")

model.eval()
with torch.no_grad():
    reconstructed_data = model(X_test_eval_torch)
    anomaly_scores = torch.mean((X_test_eval_torch - reconstructed_data) ** 2, dim=1).numpy()

benign_reconstruction_errors = torch.mean(
    (torch.tensor(X_test_benign_subset) - model(torch.tensor(X_test_benign_subset))).detach() ** 2, dim=1).numpy()
threshold = np.percentile(benign_reconstruction_errors, 85)
predicted_labels = (anomaly_scores > threshold).astype(int)

print(f"\nThreshold for anomaly detection: {threshold:.4f}")
print("\n✅ Anomaly Detection Report (0: Benign, 1: Anomaly):")
print(classification_report(y_test_eval, predicted_labels, target_names=['Benign', 'Anomaly']))
print("Confusion Matrix:")
print(confusion_matrix(y_test_eval, predicted_labels))

if len(np.unique(y_test_eval)) > 1:  # Ensure there are both classes in the test set
    auc_roc = roc_auc_score(y_test_eval, anomaly_scores)
    print(f"\nAUC-ROC Score: {auc_roc:.4f}")

    fpr, tpr, thresholds = roc_curve(y_test_eval, anomaly_scores)

    desired_tpr = 0.85
    idx = np.searchsorted(tpr, desired_tpr)  # Find index where TPR crosses desired_tpr
    if idx < len(thresholds):
        print("\n--- Exploring Thresholds for better Recall ---")
        # Print a few (FPR, TPR, Threshold) points to help manual selection
        for i in range(0, len(thresholds), len(thresholds) // 10):
            print(f"Threshold: {thresholds[i]:.4f}, FPR: {fpr[i]:.4f}, TPR: {tpr[i]:.4f}")

    optimal_idx = np.argmax(tpr - fpr)
    new_optimal_threshold = thresholds[optimal_idx]

    print(f"\nOptimal threshold (maximizing TPR-FPR): {new_optimal_threshold:.4f}")
    predicted_labels = (anomaly_scores > new_optimal_threshold).astype(int)

    print("\nAnomaly Detection Report with Optimized Threshold:")
    print(classification_report(y_test_eval, predicted_labels, target_names=['Benign', 'Anomaly']))
    print("Confusion Matrix with Optimized Threshold:")
    print(confusion_matrix(y_test_eval, predicted_labels))

    plt.figure(figsize=(8, 6)) # Create the figure
    plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (AUC = {auc_roc:.4f})')
    plt.plot([0, 1], [0, 1], color='red', lw=2, linestyle='--', label='Random Guessing')
    plt.xlabel('False Positive Rate (FPR)')
    plt.ylabel('True Positive Rate (TPR - Recall)')
    plt.title('Receiver Operating Characteristic (ROC) Curve')
    plt.grid(True)

    # Mark the chosen optimal threshold on the plot
    # Find the index closest to the new_optimal_threshold value in the 'thresholds' array
    # Note: 'thresholds' is typically sorted in decreasing order, so argmin(np.abs(thresholds - val)) works.
    idx_optimal_on_plot = np.argmin(np.abs(thresholds - new_optimal_threshold))
    plt.scatter(fpr[idx_optimal_on_plot], tpr[idx_optimal_on_plot], marker='o', color='green', s=100,
                label=f'Optimal Operating Point (Threshold={new_optimal_threshold:.4f})')

    plt.legend(loc='lower right')

    plt.savefig("working/roc_curve_optimized.png")
    print(f"✅ Optimized ROC curve saved to: working/roc_curve_optimized.png")

    plt.show()

    np.save("working/optimal_threshold.npy", new_optimal_threshold)
    print(f"✅ Optimal threshold saved to: working/optimal_threshold.npy")

# Save model
torch.save(model.state_dict(), "working/anomaly_detector_model.pth")
print(f"✅ Anomaly detector model saved to: working/anomaly_detector_model.pth")

Epoch 10/40, Loss: 0.0000


KeyboardInterrupt: 